# A/B Test Impact Simulation

This notebook estimates the potential business impact of the proposed A/B tests from Phase 3.  
The calculations are directional scenarios based on historical baseline metrics, not actual experiment results.

In [3]:
import pandas as pd
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

In [4]:
impact = pd.DataFrame([
    {
        "priority": "P1",
        "experiment_name": "Checkout Reassurance Message",
        "target_segment": "High-intent lost sessions",
        "baseline_sessions": 310249,
        "baseline_rate": 0.1241,
        "revenue_per_purchase": 40.57,
        "uplift_low": 0.03,
        "uplift_high": 0.05
    },
    {
        "priority": "P2",
        "experiment_name": "Product Recommendation Carousel",
        "target_segment": "High-exploration lost sessions",
        "baseline_sessions": 147445,
        "baseline_rate": 0.1385,
        "revenue_per_purchase": 40.57,
        "uplift_low": 0.02,
        "uplift_high": 0.04
    },
    {
        "priority": "P3",
        "experiment_name": "Vacuum Delivery Reassurance",
        "target_segment": "appliances.environment.vacuum",
        "baseline_sessions": 12410,
        "baseline_rate": 0.2575,
        "revenue_per_purchase": 39.88,
        "uplift_low": 0.03,
        "uplift_high": 0.05
    },
    {
        "priority": "P4",
        "experiment_name": "High-price Product Trust Signals",
        "target_segment": "£50+ products",
        "baseline_sessions": 324937,
        "baseline_rate": 0.0782,
        "revenue_per_purchase": 101.76,
        "uplift_low": 0.02,
        "uplift_high": 0.04
    },
    {
        "priority": "P5",
        "experiment_name": "Under £5 Checkout Friction",
        "target_segment": "Under £5 products",
        "baseline_sessions": 696183,
        "baseline_rate": 0.1793,
        "revenue_per_purchase": 16.77,
        "uplift_low": 0.02,
        "uplift_high": 0.04
    }
])

impact

,priority,experiment_name,target_segment,baseline_sessions,baseline_rate,revenue_per_purchase,uplift_low,uplift_high
0,P1,Checkout Reassurance Message,High-intent lost sessions,310249,0.1241,40.57,0.03,0.05
1,P2,Product Recommendation Carousel,High-exploration lost sessions,147445,0.1385,40.57,0.02,0.04
2,P3,Vacuum Delivery Reassurance,appliances.environment.vacuum,12410,0.2575,39.88,0.03,0.05
3,P4,High-price Product Trust Signals,£50+ products,324937,0.0782,101.76,0.02,0.04
4,P5,Under £5 Checkout Friction,Under £5 products,696183,0.1793,16.77,0.02,0.04


In [5]:
impact["baseline_conversions"] = impact["baseline_sessions"] * impact["baseline_rate"]

impact["uplift_low_rate"] = impact["baseline_rate"] * (1 + impact["uplift_low"])
impact["uplift_high_rate"] = impact["baseline_rate"] * (1 + impact["uplift_high"])

impact["incremental_purchases_low"] = (
    impact["baseline_sessions"] * impact["uplift_low_rate"]
    - impact["baseline_conversions"]
)

impact["incremental_purchases_high"] = (
    impact["baseline_sessions"] * impact["uplift_high_rate"]
    - impact["baseline_conversions"]
)

impact["incremental_revenue_low"] = (
    impact["incremental_purchases_low"] * impact["revenue_per_purchase"]
)

impact["incremental_revenue_high"] = (
    impact["incremental_purchases_high"] * impact["revenue_per_purchase"]
)

impact_summary = impact[
    [
        "priority",
        "experiment_name",
        "baseline_sessions",
        "baseline_rate",
        "baseline_conversions",
        "uplift_low",
        "uplift_high",
        "incremental_purchases_low",
        "incremental_purchases_high",
        "incremental_revenue_low",
        "incremental_revenue_high"
    ]
].copy()

impact_summary

,priority,experiment_name,baseline_sessions,baseline_rate,baseline_conversions,uplift_low,uplift_high,incremental_purchases_low,incremental_purchases_high,incremental_revenue_low,incremental_revenue_high
0,P1,Checkout Reassurance Message,310249,0.1241,38501.9009,0.03,0.05,1155.057027,1925.095045,46860.663585,78101.105976
1,P2,Product Recommendation Carousel,147445,0.1385,20421.1325,0.02,0.04,408.422650,816.845300,16569.706910,33139.413821
2,P3,Vacuum Delivery Reassurance,12410,0.2575,3195.5750,0.03,0.05,95.867250,159.778750,3823.185930,6371.976550
3,P4,High-price Product Trust Signals,324937,0.0782,25410.0734,0.02,0.04,508.201468,1016.402936,51714.581384,103429.162767
4,P5,Under £5 Checkout Friction,696183,0.1793,124825.6119,0.02,0.04,2496.512238,4993.024476,41866.510231,83733.020463


In [6]:
impact_summary = impact_summary.round({
    "baseline_rate": 4,
    "baseline_conversions": 0,
    "incremental_purchases_low": 0,
    "incremental_purchases_high": 0,
    "incremental_revenue_low": 2,
    "incremental_revenue_high": 2
})

impact_summary

,priority,experiment_name,baseline_sessions,baseline_rate,baseline_conversions,uplift_low,uplift_high,incremental_purchases_low,incremental_purchases_high,incremental_revenue_low,incremental_revenue_high
0,P1,Checkout Reassurance Message,310249,0.1241,38502.0,0.03,0.05,1155.0,1925.0,46860.66,78101.11
1,P2,Product Recommendation Carousel,147445,0.1385,20421.0,0.02,0.04,408.0,817.0,16569.71,33139.41
2,P3,Vacuum Delivery Reassurance,12410,0.2575,3196.0,0.03,0.05,96.0,160.0,3823.19,6371.98
3,P4,High-price Product Trust Signals,324937,0.0782,25410.0,0.02,0.04,508.0,1016.0,51714.58,103429.16
4,P5,Under £5 Checkout Friction,696183,0.1793,124826.0,0.02,0.04,2497.0,4993.0,41866.51,83733.02


In [7]:
power_analysis = NormalIndPower()

def calculate_sample_size(baseline_rate, relative_uplift, alpha=0.05, power=0.8):
    variant_rate = baseline_rate * (1 + relative_uplift)
    effect_size = proportion_effectsize(baseline_rate, variant_rate)
    
    sample_size = power_analysis.solve_power(
        effect_size=effect_size,
        power=power,
        alpha=alpha,
        ratio=1,
        alternative="two-sided"
    )
    
    return np.ceil(sample_size)

sample_size_rows = []

for _, row in impact.iterrows():
    for uplift_label, uplift_value in [("low", row["uplift_low"]), ("high", row["uplift_high"])]:
        sample_size_rows.append({
            "priority": row["priority"],
            "experiment_name": row["experiment_name"],
            "baseline_rate": row["baseline_rate"],
            "uplift_scenario": uplift_label,
            "relative_uplift": uplift_value,
            "variant_rate": row["baseline_rate"] * (1 + uplift_value),
            "sample_size_per_group": calculate_sample_size(
                row["baseline_rate"],
                uplift_value
            ),
            "total_sample_size": calculate_sample_size(
                row["baseline_rate"],
                uplift_value
            ) * 2
        })

sample_size = pd.DataFrame(sample_size_rows)
sample_size

,priority,experiment_name,baseline_rate,uplift_scenario,relative_uplift,variant_rate,sample_size_per_group,total_sample_size
0,P1,Checkout Reassurance Message,0.1241,low,0.03,0.127823,124681.0,249362.0
1,P1,Checkout Reassurance Message,0.1241,high,0.05,0.130305,45259.0,90518.0
2,P2,Product Recommendation Carousel,0.1385,low,0.02,0.141270,246148.0,492296.0
3,P2,Product Recommendation Carousel,0.1385,high,0.04,0.144040,62042.0,124084.0
4,P3,Vacuum Delivery Reassurance,0.2575,low,0.03,0.265225,50781.0,101562.0
5,P3,Vacuum Delivery Reassurance,0.2575,high,0.05,0.270375,18396.0,36792.0
6,P4,High-price Product Trust Signals,0.0782,low,0.02,0.079764,466820.0,933640.0
7,P4,High-price Product Trust Signals,0.0782,high,0.04,0.081328,117753.0,235506.0
8,P5,Under £5 Checkout Friction,0.1793,low,0.02,0.182886,181027.0,362054.0
9,P5,Under £5 Checkout Friction,0.1793,high,0.04,0.186472,45602.0,91204.0


In [9]:
impact_summary.to_csv("data/ab_test_impact_summary.csv", index=False)
sample_size.to_csv("data/ab_test_sample_size.csv", index=False)

## Interpretation Notes

These calculations use historical baseline conversion rates and assumed relative uplift ranges.  
They are not actual A/B test results.

Sample size estimates assume:

- Two-sided test
- 95% confidence level
- 80% statistical power
- Equal traffic split between control and variant